# 🚖 NYC Uber Pickups — Exploratory Data Analysis
**Course:** Exploratory Data Analysis  
**Dataset:** Uber NYC Pickups 2014 (April – September)  
**Features:** Date/Time, Lat, Lon, Base  
---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import glob, os, warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor']   = '#181c27'
plt.rcParams['axes.edgecolor']   = '#2a2f3e'
plt.rcParams['axes.labelcolor']  = '#c8cdd8'
plt.rcParams['xtick.color']      = '#c8cdd8'
plt.rcParams['ytick.color']      = '#c8cdd8'
plt.rcParams['text.color']       = '#e8e3d5'
plt.rcParams['grid.color']       = '#2a2f3e'
plt.rcParams['grid.linewidth']   = 0.5
print('Libraries loaded!')

## 2. Load & Combine Dataset

In [ ]:
files = sorted(glob.glob('../data/uber-raw-data-*.csv'))
print(f'Found {len(files)} files:')
for f in files:
    print(' ', f)

dfs = [pd.read_csv(f) for f in files]
df  = pd.concat(dfs, ignore_index=True)
print(f'\nTotal rows: {len(df):,}')

## 3. Data Inspection

In [ ]:
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head()

In [ ]:
print('Data types:\n', df.dtypes)
print('\nMissing values:\n', df.isnull().sum())

In [ ]:
df.describe()

## 4. Data Cleaning & Feature Engineering

In [ ]:
df.columns = ['DateTime', 'Lat', 'Lon', 'Base']
df['DateTime'] = pd.to_datetime(df['DateTime'])

# Remove GPS outliers outside NYC
df = df[(df['Lat'] > 40.4) & (df['Lat'] < 41.0)]
df = df[(df['Lon'] > -74.3) & (df['Lon'] < -73.6)]
df.dropna(inplace=True)

# Feature engineering
df['Date']    = df['DateTime'].dt.date
df['Month']   = df['DateTime'].dt.month
df['Hour']    = df['DateTime'].dt.hour
df['Weekday'] = df['DateTime'].dt.weekday
day_names     = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
df['DayName'] = df['Weekday'].map(lambda x: day_names[x])
df['Week']    = df['DateTime'].dt.isocalendar().week.astype(int)

print(f'Clean dataset shape: {df.shape}')
df.head()

## 5. Descriptive Statistics

In [ ]:
month_names = {4:'Apr',5:'May',6:'Jun',7:'Jul',8:'Aug',9:'Sep'}
day_order   = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_short   = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

print('=== KEY STATISTICS ===')
print(f'Total pickups  : {len(df):,}')
print(f'Date range     : {df["DateTime"].min()} to {df["DateTime"].max()}')
print(f'Unique bases   : {df["Base"].nunique()}')
print(f'Peak hour      : {df["Hour"].mode()[0]:02d}:00')
print(f'Busiest day    : {df["DayName"].mode()[0]}')
print(f'\nPickups per base:\n{df["Base"].value_counts()}')

## 6. Chart 1 — Bar Chart: Pickups by Hour

In [ ]:
hourly = df.groupby('Hour').size().reindex(range(24), fill_value=0)
fig, ax = plt.subplots(figsize=(12, 4))
colors = ['#e8533a' if h == hourly.idxmax() else '#e8533a55' for h in range(24)]
ax.bar(hourly.index, hourly.values, color=colors, width=0.8, zorder=2)
ax.set_xlabel('Hour of Day'); ax.set_ylabel('Pickups')
ax.set_title('Pickups by Hour of Day', fontsize=14)
ax.set_xticks(range(0, 24))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
ax.grid(axis='y', alpha=0.4)
plt.tight_layout(); plt.show()
print(f'Peak hour: {hourly.idxmax()}:00 with {hourly.max():,} pickups')

## 7. Chart 2 — Histogram: Hour Frequency Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df['Hour'], bins=24, range=(0,24), color='#e8533a', alpha=0.85, edgecolor='#0f1117', zorder=2)
ax.set_xlabel('Hour of Day'); ax.set_ylabel('Frequency')
ax.set_title('Frequency Distribution of Pickup Hours (Histogram)', fontsize=14)
ax.grid(axis='y', alpha=0.4)
plt.tight_layout(); plt.show()

## 8. Chart 3 — Line Chart: Monthly Trend

In [ ]:
monthly = df.groupby('Month').size()
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly.index, monthly.values, color='#e8533a', linewidth=2.5,
        marker='o', markersize=8, markerfacecolor='#0f1117', markeredgecolor='#e8533a', zorder=3)
ax.fill_between(monthly.index, monthly.values, alpha=0.15, color='#e8533a')
ax.set_xticks(monthly.index)
ax.set_xticklabels([month_names[m] for m in monthly.index])
ax.set_xlabel('Month'); ax.set_ylabel('Total Pickups')
ax.set_title('Monthly Pickup Trend (Line Chart)', fontsize=14)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
ax.grid(alpha=0.4)
plt.tight_layout(); plt.show()

## 9. Chart 4 — Bar Chart: Pickups by Weekday

In [ ]:
wd = df.groupby('DayName').size().reindex(day_order, fill_value=0)
fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#e8533a' if d == wd.idxmax() else '#3a7bd599' for d in day_order]
ax.bar(day_short, wd.values, color=colors, zorder=2)
ax.set_xlabel('Day of Week'); ax.set_ylabel('Pickups')
ax.set_title('Pickups by Day of Week (Bar Chart)', fontsize=14)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
ax.grid(axis='y', alpha=0.4)
plt.tight_layout(); plt.show()

## 10. Chart 5 — Scatter Plot: Pickup Locations

In [ ]:
sample = df.sample(10000, random_state=42)
fig, ax = plt.subplots(figsize=(8, 7))
sc = ax.scatter(sample['Lon'], sample['Lat'], alpha=0.2, s=2,
                c=sample['Hour'], cmap='inferno', zorder=2)
plt.colorbar(sc, ax=ax, label='Hour of Day')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('Pickup Locations by Hour (Scatter Plot)', fontsize=14)
ax.set_xlim(-74.3, -73.65); ax.set_ylim(40.45, 40.95)
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 11. Chart 6 — Box Plot: Hour Distribution by Base

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
bases = sorted(df['Base'].unique())
data  = [df[df['Base'] == b]['Hour'].values for b in bases]
bp = ax.boxplot(data, labels=bases, patch_artist=True,
                medianprops=dict(color='#e8533a', linewidth=2),
                whiskerprops=dict(color='#c8cdd8'),
                capprops=dict(color='#c8cdd8'),
                flierprops=dict(marker='.', color='#555d70', markersize=3, alpha=0.4))
for patch in bp['boxes']:
    patch.set_facecolor('#181c27'); patch.set_edgecolor('#2a2f3e')
ax.set_xlabel('Base'); ax.set_ylabel('Hour of Day')
ax.set_title('Hour Distribution by Base (Box Plot)', fontsize=14)
ax.grid(axis='y', alpha=0.4)
plt.tight_layout(); plt.show()

## 12. Chart 7 — Heatmap: Day x Hour

In [ ]:
pivot = df.groupby(['DayName','Hour']).size().unstack(fill_value=0).reindex(day_order)
fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(pivot, ax=ax, cmap='YlOrRd', linewidths=0.3, linecolor='#0f1117',
            cbar_kws={'label':'Pickups'}, xticklabels=2)
ax.set_title('Pickups Heatmap - Day x Hour', fontsize=14)
ax.set_xlabel('Hour of Day'); ax.set_ylabel('')
plt.tight_layout(); plt.show()

## 13. Chart 8 — Area Chart: Cumulative Pickups

In [ ]:
cumulative = df.groupby('Month').size().cumsum()
fig, ax = plt.subplots(figsize=(10, 4))
ax.fill_between(cumulative.index, cumulative.values, alpha=0.4, color='#4caf80')
ax.plot(cumulative.index, cumulative.values, color='#4caf80', linewidth=2.5,
        marker='o', markersize=7, markerfacecolor='#0f1117', markeredgecolor='#4caf80')
ax.set_xticks(cumulative.index)
ax.set_xticklabels([month_names[m] for m in cumulative.index])
ax.set_xlabel('Month'); ax.set_ylabel('Cumulative Pickups')
ax.set_title('Cumulative Pickups Over Time (Area Chart)', fontsize=14)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
ax.grid(alpha=0.4)
plt.tight_layout(); plt.show()

## 14. Chart 9 — Count Plot: Rides per Base

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
order  = df['Base'].value_counts().index.tolist()
counts = df['Base'].value_counts()
colors = ['#e8533a' if i == 0 else '#3a7bd599' for i in range(len(order))]
ax.barh(order[::-1], counts.reindex(order).values[::-1], color=colors[::-1], zorder=2)
ax.set_xlabel('Count')
ax.set_title('Ride Count by Dispatch Base (Count Plot)', fontsize=14)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
ax.grid(axis='x', alpha=0.4)
plt.tight_layout(); plt.show()

## 15. Chart 10 — Violin Plot: Hour by Weekday

In [ ]:
sample = df.sample(min(30000, len(df)), random_state=1)
fig, ax = plt.subplots(figsize=(12, 5))
palette = ['#e8533a','#e8533a','#3a7bd5','#3a7bd5','#e8533a','#4caf80','#4caf80']
sns.violinplot(data=sample, x='DayName', y='Hour', order=day_order,
               ax=ax, palette=palette, inner='quartile', linewidth=0.8, cut=0)
ax.set_xlabel('Day of Week'); ax.set_ylabel('Hour of Day')
ax.set_title('Hour Distribution by Weekday (Violin Plot)', fontsize=14)
ax.grid(axis='y', alpha=0.4)
plt.tight_layout(); plt.show()

## 16. Summary & Key Insights

In [ ]:
print('=' * 50)
print('        KEY INSIGHTS SUMMARY')
print('=' * 50)
print(f'Total Pickups     : {len(df):,}')
print(f'Peak Hour         : {df["Hour"].mode()[0]:02d}:00')
print(f'Busiest Day       : {df["DayName"].mode()[0]}')
print(f'Busiest Month     : {month_names[df["Month"].mode()[0]]}')
print(f'Top Dispatch Base : {df["Base"].mode()[0]}')
print()
print('Pickups by Base:')
for base, cnt in df['Base'].value_counts().items():
    pct = cnt/len(df)*100
    print(f'  {base}: {cnt:,} ({pct:.1f}%)')